# 04_HFACS_Classification

This notebook maps extracted human-factor and contributing-factor signals into HFACS levels:

1. Unsafe Acts  
2. Preconditions for Unsafe Acts  
3. Unsafe Supervision  
4. Organizational Influences

In [1]:
import pandas as pd
import numpy as np

In [2]:
# Load offline-extracted dataset
df = pd.read_csv("../data/processed/asrs_extracted_offline_3yrs.csv")

print("Loaded:", df.shape)
df[['human_factors', 'contributing_factors', 'risk_signals']].head()

Loaded: (16535, 131)


C:\Users\jenny\AppData\Local\Temp\ipykernel_13684\1852849555.py:2: DtypeWarning: Columns (7,8,15,19,20,38,39,40,41,42,43,44,45,46,47,48,49,50,59,63,78,79,81,82,83,86,89,99,100,110,111,123) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/asrs_extracted_offline_3yrs.csv")


,human_factors,contributing_factors,risk_signals
0,['decision_errors'],['workload'],NaN
1,['skill_based_errors'],['automation'],NaN
2,NaN,['weather'],NaN
3,NaN,NaN,NaN
4,NaN,NaN,NaN


In [7]:
# Core HFACS levels
HFACS_LEVELS = {
    "unsafe_acts": [
        "decision_errors", "skill_based_errors", "violations", "automation"
    ],
    "preconditions": [
        "fatigue", "workload", "situational_awareness", "communication", "weather"
    ],
    "unsafe_supervision": [
        "inadequate_supervision", "failed_to_correct_problem", "planned_inappropriate_operations"
    ],
    "organizational_influences": [
        "organizational_climate", "resource_management", "organizational_processes"
    ],
}

# Keyword expansions (from contributing_factors / risk_signals)
HFACS_KEYWORDS = {
    "unsafe_acts": ["incorrect", "mistake", "failed", "overshoot", "misjudged", "automation"],
    "preconditions": ["tired", "fatigue", "busy", "task saturated", "high workload", "did not see", "weather"],
    "unsafe_supervision": ["training", "supervision", "checklist not followed"],
    "organizational_influences": ["policy", "procedure", "company", "management"],
}

In [8]:
def to_list(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    # handle stringified lists like "['a', 'b']"
    if isinstance(x, str):
        return [x.lower()]
    return []

In [9]:
def classify_hfacs(row):
    human = to_list(row.get("human_factors"))
    contrib = to_list(row.get("contributing_factors"))
    risks = to_list(row.get("risk_signals"))

    text_blob = " ".join(human + contrib + risks).lower()

    levels = {
        "unsafe_acts": False,
        "preconditions": False,
        "unsafe_supervision": False,
        "organizational_influences": False,
    }

    # Rule-based: direct matches from human_factors
    for hf in human:
        hf = hf.lower()
        if hf in HFACS_LEVELS["unsafe_acts"]:
            levels["unsafe_acts"] = True
        if hf in HFACS_LEVELS["preconditions"]:
            levels["preconditions"] = True

    # Keyword-based: expanded patterns from text_blob
    for level, keywords in HFACS_KEYWORDS.items():
        if any(k in text_blob for k in keywords):
            levels[level] = True

    return pd.Series(levels)

In [10]:
hfacs_cols = df.apply(classify_hfacs, axis=1)

for col in hfacs_cols.columns:
    df[f"HFACS_{col}"] = hfacs_cols[col]

df[[c for c in df.columns if c.startswith("HFACS_")]].head()

,HFACS_unsafe_acts,HFACS_preconditions,HFACS_unsafe_supervision,HFACS_organizational_influences
0,False,False,False,False
1,True,False,False,False
2,False,True,False,False
3,False,False,False,False
4,False,False,False,False


In [11]:
output_file = "../data/processed/asrs_hfacs_3yrs.csv"
df.to_csv(output_file, index=False)

print("Saved HFACS dataset to:", output_file)
print("Final shape:", df.shape)

Saved HFACS dataset to: ../data/processed/asrs_hfacs_3yrs.csv
Final shape: (16535, 135)
